# Práctica: pedidos con Kafka y HDFS

## 1. Contexto

Una empresa de comercio electrónico recibe continuamente pedidos de sus clientes. Para gestionar esta información necesita registrar los pedidos, transmitirlos mediante Apache Kafka y almacenarlos en HDFS para su consulta y análisis posteriores.

## 2. Objetivo

Desarrollar un flujo completo de datos que lea los pedidos JSON desde HDFS, los envíe individualmente a Kafka, los consuma y escriba un nuevo archivo JSON en HDFS.

```
HDFS (entrada)  →  Productor Python  →  Kafka  →  Consumidor Python  →  HDFS (salida)
```

## Preparación del entorno

Instalamos las librerías necesarias: `confluent-kafka` para el productor y el consumidor, y `hdfs` para trabajar con HDFS desde Python.

In [ ]:
%pip install confluent-kafka hdfs

Definimos los datos de conexión (broker de Kafka y servidor HDFS), el tópico, el grupo de consumidores y las rutas de los archivos.

In [1]:
import json
import socket
import time
from hdfs import InsecureClient

USUARIO = "iabd-user18"

KAFKA_BROKER = "10.22.87.215:9092"
HDFS_URL = "http://10.22.87.215:50070"

TOPIC = f"pedidos_{USUARIO}"
GROUP_ID = f"grupo_pedidos_{USUARIO}_01"
TIMEOUT = 30

DIR_ENTRADA = f"/user/{USUARIO}/pedidos/entrada"
DIR_SALIDA = f"/user/{USUARIO}/pedidos/salida"
HDFS_ENTRADA = f"{DIR_ENTRADA}/pedidos.json"
HDFS_SALIDA = f"{DIR_SALIDA}/pedidos_recibidos.json"

client = InsecureClient(HDFS_URL, user=USUARIO)

## Actividad 1. Preparación de los datos

Partimos del archivo `pedidos.json`, que contiene un array JSON con los pedidos. Cada pedido tiene los campos `id_pedido`, `cliente`, `producto`, `categoria`, `cantidad`, `precio_unitario` y `fecha`.

Creamos la carpeta de entrada en la carpeta personal de HDFS y subimos el archivo.

In [ ]:
client.makedirs(DIR_ENTRADA)
client.upload(HDFS_ENTRADA, "pedidos.json", overwrite=True)

print(client.list(DIR_ENTRADA))

['pedidos.json']


## Actividad 2. Productor Kafka

El productor se conecta al broker de Kafka y lee `pedidos.json` desde HDFS. Después envía cada pedido como un mensaje JSON independiente al tópico `pedidos_<usuario>`, usando `id_pedido` como clave. La función `confirmar` muestra por pantalla si el mensaje se ha entregado o si ha habido un error.

In [3]:
from confluent_kafka import Producer

# Leer los pedidos desde HDFS
with client.read(HDFS_ENTRADA, encoding="utf-8") as reader:
    pedidos = json.load(reader)

print("Pedidos leídos de HDFS:", len(pedidos))

# Conectar con el broker
producer = Producer({
    "bootstrap.servers": KAFKA_BROKER,
    "client.id": socket.gethostname()
})

def confirmar(err, msg):
    if err:
        print("Error al enviar:", err)
    else:
        print("Enviado:", msg.key().decode("utf-8"), msg.value().decode("utf-8"))

# Enviar cada pedido como un mensaje independiente
for pedido in pedidos:
    producer.produce(
        TOPIC,
        key=str(pedido["id_pedido"]),
        value=json.dumps(pedido, ensure_ascii=False),
        callback=confirmar
    )

producer.flush(15)

Pedidos leídos de HDFS: 20
Enviado: 1 {"id_pedido": 1, "cliente": "Ane", "producto": "Portátil", "categoria": "Informática", "cantidad": 1, "precio_unitario": 850, "fecha": "2026-10-19"}
Enviado: 2 {"id_pedido": 2, "cliente": "Jon", "producto": "Ratón", "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 25, "fecha": "2026-10-19"}
Enviado: 3 {"id_pedido": 3, "cliente": "Maite", "producto": "Teclado mecánico", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 70, "fecha": "2026-10-15"}
Enviado: 4 {"id_pedido": 4, "cliente": "Iker", "producto": "Monitor 27 pulgadas", "categoria": "Informática", "cantidad": 2, "precio_unitario": 210, "fecha": "2026-10-15"}
Enviado: 5 {"id_pedido": 5, "cliente": "Leire", "producto": "Auriculares Bluetooth", "categoria": "Audio", "cantidad": 1, "precio_unitario": 59, "fecha": "2026-10-15"}
Enviado: 6 {"id_pedido": 6, "cliente": "Ander", "producto": "Smartphone", "categoria": "Telefonía", "cantidad": 1, "precio_unitario": 599, "fecha": "2

0

## Actividad 3. Consumidor Kafka

El consumidor se suscribe al tópico con un grupo de consumidores propio. Cada mensaje recibido se convierte de JSON a diccionario Python, se muestra por pantalla y se guarda en una lista.

Para controlar los errores, se comprueba `msg.error()` en cada mensaje. Como tiempo máximo de espera, el consumidor se detiene si pasan `TIMEOUT` segundos sin recibir ningún mensaje nuevo, o cuando ya ha recibido todos los pedidos.

In [4]:
from confluent_kafka import Consumer

consumer = Consumer({
    "bootstrap.servers": KAFKA_BROKER,
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest"
})
consumer.subscribe([TOPIC])

pedidos_recibidos = []
ultimo_mensaje = time.time()

try:
    while len({p["id_pedido"] for p in pedidos_recibidos}) < len(pedidos) \
            and time.time() - ultimo_mensaje < TIMEOUT:

        msg = consumer.poll(timeout=2.0)

        if msg is None:
            continue

        if msg.error():
            print("Error:", msg.error())
            continue

        pedido = json.loads(msg.value().decode("utf-8"))
        pedidos_recibidos.append(pedido)
        ultimo_mensaje = time.time()
        print("Recibido:", pedido)

finally:
    consumer.close()

Recibido: {'id_pedido': 1, 'cliente': 'Ane', 'producto': 'Portátil', 'categoria': 'Informática', 'cantidad': 1, 'precio_unitario': 850, 'fecha': '2026-10-19'}
Recibido: {'id_pedido': 2, 'cliente': 'Jon', 'producto': 'Ratón', 'categoria': 'Accesorios', 'cantidad': 2, 'precio_unitario': 25, 'fecha': '2026-10-19'}
Recibido: {'id_pedido': 3, 'cliente': 'Maite', 'producto': 'Teclado mecánico', 'categoria': 'Accesorios', 'cantidad': 1, 'precio_unitario': 70, 'fecha': '2026-10-15'}
Recibido: {'id_pedido': 4, 'cliente': 'Iker', 'producto': 'Monitor 27 pulgadas', 'categoria': 'Informática', 'cantidad': 2, 'precio_unitario': 210, 'fecha': '2026-10-15'}
Recibido: {'id_pedido': 5, 'cliente': 'Leire', 'producto': 'Auriculares Bluetooth', 'categoria': 'Audio', 'cantidad': 1, 'precio_unitario': 59, 'fecha': '2026-10-15'}
Recibido: {'id_pedido': 6, 'cliente': 'Ander', 'producto': 'Smartphone', 'categoria': 'Telefonía', 'cantidad': 1, 'precio_unitario': 599, 'fecha': '2026-10-16'}
Recibido: {'id_pedido

Comprobamos que se han recuperado todos los pedidos.

In [5]:
print(f"Pedidos recibidos: {len(pedidos_recibidos)} de {len(pedidos)}")

Pedidos recibidos: 20 de 20


## Actividad 4. Escritura en HDFS

Para evitar pérdidas y duplicados, primero eliminamos los pedidos repetidos usando `id_pedido` y comparamos los identificadores recibidos con los originales.

In [6]:
# Quitar duplicados (un pedido por id)
unicos = {p["id_pedido"]: p for p in pedidos_recibidos}
pedidos_finales = list(unicos.values())

# Comparar los identificadores recibidos con los originales
ids_originales = {p["id_pedido"] for p in pedidos}
print("Identificadores que faltan:", ids_originales - set(unicos))

Identificadores que faltan: set()


Guardamos los pedidos en `pedidos_recibidos.json` dentro de la carpeta de salida de HDFS, manteniendo el formato original: un array de objetos con los mismos campos.

In [7]:
client.makedirs(DIR_SALIDA)

with client.write(HDFS_SALIDA, encoding="utf-8", overwrite=True) as writer:
    json.dump(pedidos_finales, writer, ensure_ascii=False, indent=2)

print(client.list(DIR_SALIDA))

['pedidos_recibidos.json']


Para verificar la escritura, leemos el archivo desde HDFS y comprobamos que contiene todos los pedidos.

In [8]:
with client.read(HDFS_SALIDA, encoding="utf-8") as reader:
    pedidos_hdfs = json.load(reader)

print("Pedidos en el archivo:", len(pedidos_hdfs))
print("Mismos identificadores que los originales:",
      {p["id_pedido"] for p in pedidos_hdfs} == ids_originales)

Pedidos en el archivo: 20
Mismos identificadores que los originales: True


## Actividad 5. Lectura y comprobación desde HDFS

Leemos `pedidos_recibidos.json` directamente desde HDFS y, sin utilizar Pandas, calculamos el número total de pedidos, el importe total (`cantidad × precio_unitario`), el pedido con mayor importe, el número de pedidos por categoría y la comprobación de los identificadores originales.

In [9]:
with client.read(HDFS_SALIDA, encoding="utf-8") as reader:
    datos = json.load(reader)

# Número total de pedidos
print("Total de pedidos:", len(datos))

# Importe total
importe_total = sum(p["cantidad"] * p["precio_unitario"] for p in datos)
print("Importe total:", importe_total)

# Pedido con mayor importe
mayor = max(datos, key=lambda p: p["cantidad"] * p["precio_unitario"])
print("Pedido con mayor importe:", mayor)

# Número de pedidos por categoría
por_categoria = {}
for p in datos:
    por_categoria[p["categoria"]] = por_categoria.get(p["categoria"], 0) + 1
print("Pedidos por categoría:", por_categoria)

# Todos los identificadores originales aparecen en el archivo final
with client.read(HDFS_ENTRADA, encoding="utf-8") as reader:
    ids_originales = {p["id_pedido"] for p in json.load(reader)}
ids_finales = {p["id_pedido"] for p in datos}
print("Todos los identificadores originales aparecen:", ids_originales <= ids_finales)

Total de pedidos: 20
Importe total: 4680
Pedido con mayor importe: {'id_pedido': 20, 'cliente': 'Lucia', 'producto': 'Portátil', 'categoria': 'Informática', 'cantidad': 2, 'precio_unitario': 780, 'fecha': '2026-10-19'}
Pedidos por categoría: {'Informática': 6, 'Accesorios': 5, 'Audio': 3, 'Telefonía': 3, 'Almacenamiento': 3}
Todos los identificadores originales aparecen: True
